# Linear Regression

## 1. Single Variable

The equation of the regression line is:  
$$y = mx + b$$  

The residual error in the observation is:  
$$\epsilon_i = y_i - mx_i - b$$

The aim is to minimize the total residual error. We define the squared error or cost function, $J$ as:  
$$J(m, b) = \frac{1}{2n} \sum^n_{i=1}{\epsilon_i^2}$$  

Our task is to find value of $m$ and $b$ for which $J(m, b)$ is minimum

$$m = \frac{\sum{(x - \bar{x})(y - \bar{y})}}{\sum{(x - \bar{x})^2}}$$  
$$b = \bar{y} - m\bar{x}$$

Multiple linear regression is a more general form that models the relationship between multiple independent variables and one dependent variable

$$y = w_0 + w_1x_1 + w_2x_2 + ... + w_nx_n = X^TW$$

In [43]:
import numpy as np

class LinearRegression:
    def __init__(self):
        self.m = None
        self.b = None 

    def fit(self, x, y): #find m and b
        x_mean = np.mean(x)
        y_mean = np.mean(y)
        numerator = np.sum((x - x_mean) * (y - y_mean))
        denominator = np.sum((x - x_mean)**2)
        self.m = numerator / denominator 
        self.b = y_mean - self.m * x_mean

    def predict(self, x):
        return self.m * x + self.b


In [44]:
x = np.array([1, 2, 3, 4, 5])
y = 2*x + 3 + [0.01, -0.01, 0.02, 0.0, -0.02]


LR = LinearRegression()
LR.fit(x, y)
y_pred = LR.predict(x)

y_pred

array([ 5.01 ,  7.005,  9.   , 10.995, 12.99 ])

## 2. Multi-dimensional data

Linear regression with multiple features: 
Model: 
$$\hat{y} = Xw + b, \quad X\in \mathbb{R}^{n\times d}, y\in \mathbb{R}^n, w\in \mathbb{R}^d, b\in \mathbb{R}$$
Augment $X$ with a first column of ones:
$$\tilde{X} = [1 \quad X]$$
$$\theta = \begin{bmatrix} b \\ w \end{bmatrix}$$

Then:

$$\hat{y} = \tilde{X}\theta$$

The closed-form least squares:
$$\theta = (\tilde{X}^T\tilde{X})^{-1}\tilde{X}^Ty$$


### 2.1 Gradient Descent

We minimize mean squared error:
$$J(w, b) = \frac{1}{n} \sum^n_{i=1}(x_i^Tw + b - y_i)^2$$

let:
$$\hat{y} = Xw + b$$
Then gradient w.r.t weights and gradient w.r.t bias can be calculated as:
$$\nabla_w = \frac{2}{n} X^T(\hat{y} - y)$$
$$\nabla_b = \frac{2}{n}\sum(\hat{y} - y)$$

In [119]:
class LinearRegressionGD:
    def __init__(self, lr=0.001, max_iter=50):
        self.w = None
        self.b = None
        self.lr = lr
        self.max_iter = max_iter

    def fit(self, X, y):
        n, d = X.shape
        self.w = np.zeros(d)
        self.b = 0.0
        
        for it in range(self.max_iter):
            y_pred = X@self.w + self.b
            error = y_pred - y
            dw = 2/n * X.T @ error 
            db = 2/n * np.sum(error)
            self.w -= self.lr * dw
            self.b -= self.lr * db 

    def predict(self, X):
        return X@self.w + self.b
            
            

In [121]:
#X=(n, d), y=(n,), w=(n,)
# the shape of y needs to match w!! 

X = np.random.rand(3,4)
y = np.random.rand(3)
lrgd = LinearRegressionGD()
lrgd.fit(X, y)
y_hat = lrgd.predict(X)
y - y_hat

array([0.68107888, 0.58084114, 0.41106245])

### 2.2 Closed-form Solution

In [92]:
class LinearRegressionND:
    def __init__(self):
        self.theta = None

    def fit(self, X, y):
        n, d = X.shape
        ones = np.ones((n, 1))
        X = np.hstack((ones, X))
        # XT_X = X.T @ X
        # XT_y = X.T @ y
        self.theta, *_ = np.linalg.lstsq(X, y, rcond=None)


    def predict(self, X):
        n, d = X.shape
        ones = np.ones((n, 1))
        X = np.hstack((ones, X))
        return X@self.theta
        
        

In [93]:
X = np.random.rand(3, 4)
y = np.random.rand(3,1)

lrnd = LinearRegressionND()
lrnd.fit(X, y)
lrnd.predict(X)

array([[0.21124603],
       [0.61838962],
       [0.37254965]])

**Q: Why not using the closed form function?**

Because computing $(X^TX)^{-1}$ numerically worse, especially when the matrix is singular or rank deficient, inversion is unstable. 

Lstsq uses SVD to compute the least-squares solution. 